# 04. Regressors and weather

A regressor is an extra column that explains part of your series, such as a promotion flag, a price or the temperature. This notebook shows how to hand `forecast()` regressors, what it needs from you for the future, how to read what each regressor contributes, and how `weather=` fetches local temperature for you.

Sections 7 to 9 use the network: they call the [Open-Meteo](https://open-meteo.com) API (no key needed, responses cached on disk). Open-Meteo's free API is for **non-commercial use only**, subject to its [terms](https://open-meteo.com/en/terms). Everything before section 7 runs offline.

- What a regressor is, and when it helps
- Regressors from columns, with future values via `future=`
- The dict-of-Series form
- `regressor_options`: multiplicative mode and prior scale
- When future values are missing
- What each regressor contributes: `plot_regressor_effects()`
- Local temperature with `weather=`
- A recent series: weather forecast, then climatology
- When a temperature regressor does not help

Previous: [03. Seasonality and holidays](03_seasonality_and_holidays.ipynb) | Next: [05. Uncertainty and outliers](05_uncertainty_and_outliers.ipynb)

In [ ]:
from dataclasses import asdict

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from prophet import forecast
from prophet.quick import MissingRegressorValuesError
from prophet.utilities import regressor_coefficients
from prophet.weather import fetch_temperature

from helpers import load_example, quiet_logs

quiet_logs()

## 1. What a regressor is, and when it helps

Prophet models trend, seasonality and holidays. A regressor adds something it cannot know from the calendar alone, as long as you know its value for every day you forecast. Here the idea is an NFL Sunday flag for Peyton Manning's Wikipedia page views: Sundays in the football season (September to January) should differ from Sundays out of season, which a plain weekly pattern cannot express.

In [ ]:
def nfl_sunday(ds: pd.Series) -> pd.Series:
    """1.0 on Sundays from September to January (the NFL season), otherwise 0.0."""
    in_season = (ds.dt.month >= 9) | (ds.dt.month <= 1)
    return ((ds.dt.dayofweek == 6) & in_season).astype(float)


df = load_example("peyton_manning")
df["nfl_sunday"] = nfl_sunday(df["ds"])
df.tail()

In [ ]:
day_type = np.select(
    [df["nfl_sunday"] == 1, df["ds"].dt.dayofweek == 6],
    ["Sunday in season", "Sunday out of season"],
    default="Other days",
)
df.groupby(day_type)["y"].agg(["mean", "count"]).round(2)

**Takeaway:** the values are log page views, so in-season Sundays sit visibly above both other groups. That gap is what the regressor can learn.

## 2. Regressors from columns, with future values via `future=`

Name the columns in `regressors=`. The history comes from the data, so `forecast()` also needs the same column for the forecast dates, which you pass as `future=`: a frame with a `ds` column and one column per regressor. This series skips a few days, so `freq="D"` is given explicitly rather than inferred.

In [ ]:
HORIZON = "365D"
FREQ = "D"

start = df["ds"].max() + pd.Timedelta(days=1)
future = pd.DataFrame({"ds": pd.date_range(start, periods=365, freq=FREQ)})
future["nfl_sunday"] = nfl_sunday(future["ds"])

result = forecast(df, HORIZON, freq=FREQ, regressors=["nfl_sunday"], future=future)
print(result.summary())
result.future.head()

In [ ]:
result.regressor_values.tail()

`result.regressor_values` keeps the regressor for every training and forecast row, so you can check exactly what the model was given.

## 3. The dict-of-Series form

When the regressor does not live in your data frame, pass a dict of Series indexed by date. Each Series must cover the history and the forecast dates, so no `future=` is needed. A DataFrame with a `ds` column works the same way.

In [ ]:
calendar = pd.date_range(df["ds"].min(), future["ds"].max(), freq=FREQ)
nfl_series = pd.Series(nfl_sunday(pd.Series(calendar)).to_numpy(), index=calendar)

result_dict = forecast(
    df[["ds", "y"]], HORIZON, freq=FREQ, regressors={"nfl_sunday": nfl_series}
)
print("Same forecast as section 2:", np.allclose(result.future["yhat"], result_dict.future["yhat"]))

## 4. `regressor_options`: multiplicative mode and prior scale

By default a regressor adds a fixed amount per unit. `{"mode": "multiplicative"}` makes its effect a share of the trend instead, and `prior_scale` (default 10) sets how strongly the model is allowed to use it: smaller values shrink the coefficient. Here each variant is scored on a backtest of the last 365 days.

In [ ]:
def compare_regressor_settings(variants: dict) -> pd.DataFrame:
    """Coefficient and backtest scores for each regressor setting (None means no regressor)."""
    rows = []
    for label, options in variants.items():
        if options is None:
            res = forecast(df[["ds", "y"]], HORIZON, freq=FREQ)
            coef = np.nan
        else:
            res = forecast(
                df, HORIZON, freq=FREQ, regressors=["nfl_sunday"], future=future,
                regressor_options={"nfl_sunday": options},
            )
            coef = regressor_coefficients(res.model)["coef"].iloc[0]
        scores = res.backtest(HORIZON).metrics
        rows.append({"variant": label, "coef": coef, "MAE": scores.mae,
                     "RMSE": scores.rmse, "coverage": scores.coverage})
    return pd.DataFrame(rows).set_index("variant").round(3)


compare_regressor_settings({
    "no regressor": None,
    "additive (default)": {},
    "multiplicative": {"mode": "multiplicative"},
    "additive, prior_scale=0.1": {"prior_scale": 0.1},
})

**Takeaway:** the regressor helps a little, and the options barely matter here. The additive coefficient is in log units, the multiplicative one is a share of the trend, and a small `prior_scale` changes little because 2,900 days of data outweigh the prior. Expect options to matter more with short or noisy data.

## 5. When future values are missing

A regressor you cannot supply for the future cannot be forecast with. `forecast()` refuses to guess and names the dates that are missing, instead of failing deep inside Stan.

In [ ]:
try:
    forecast(df, HORIZON, freq=FREQ, regressors=["nfl_sunday"])
except MissingRegressorValuesError as error:
    print(f"{type(error).__name__}: {error}")

## 6. What each regressor contributes

`result.plot_regressor_effects()` draws one panel per regressor: how much it moved the forecast over time, with a card that says what one extra unit does in plain words.

In [ ]:
fig = result.plot_regressor_effects()

**Takeaway:** the flag is 0 or 1, so the contribution jumps by the same amount on every in-season Sunday and is flat otherwise.

## 7. Local temperature with `weather=`

Temperature is a regressor you cannot build from a column, so `weather="Melbourne, Australia"` fetches it for you and adds it as a regressor named `temperature`, for the history and the horizon. The data is daily pedestrian counts in central Melbourne, cut before 2020 so the lockdowns stay out of the way.

The first call needs the network and is cached on disk afterwards (set `NEOPROPHET_CACHE_DIR` to choose where). The 60-day horizon here is January and February 2020, which is already in the past, so every value comes from Open-Meteo's archive of measured weather.

In [ ]:
pedestrians = load_example("pedestrians")
train = pedestrians[pedestrians["ds"] < "2020-01-01"]

result = forecast(train, "60D", weather="Melbourne, Australia")
print(result.summary())
print(result.weather.location)
result.weather.frame.head()

`result.weather.frame` has one row per day with the temperature in °C and a `source` column that records where each value came from: `archive` (measured), `forecast` (the weather forecast) or `climatology` (typical for the date). Counting them shows what the model was given.

In [ ]:
result.weather.frame["source"].value_counts()

In [ ]:
fig = result.plot_regressor_effects()

In [ ]:
coef = regressor_coefficients(result.model)["coef"].iloc[0]
mean_y = train["y"].mean()
print(f"Fitted effect: {coef:,.0f} pedestrians per extra degree, against a mean of {mean_y:,.0f} a day ({coef / mean_y:+.2%}).")

Now the real test: the same 60 days that really happened, forecast with and without the temperature. The held-out dates are in the data, so both forecasts can be scored.

In [ ]:
def holdout_scores(res, actual: pd.DataFrame) -> pd.Series:
    """MAE and interval coverage of a forecast against what really happened."""
    joined = res.future.merge(actual, on="ds")
    return pd.Series({
        "days": len(joined),
        "MAE": (joined["yhat"] - joined["y"]).abs().mean(),
        "coverage": joined["y"].between(joined["yhat_lower"], joined["yhat_upper"]).mean(),
    })


actual = pedestrians[pedestrians["ds"].between(result.future["ds"].min(), result.future["ds"].max())]
without_weather = forecast(train, "60D")
pd.DataFrame({
    "without temperature": holdout_scores(without_weather, actual),
    "with temperature": holdout_scores(result, actual),
}).T.round(3)

**Takeaway:** for central Melbourne pedestrian counts the temperature effect is tiny and the held-out error is no better with it. The mechanics work; this series just does not depend much on temperature. Section 8 uses a series that does.

## 8. A recent series: weather forecast, then climatology

Real forecasts look ahead of today, and then the future temperature is itself a forecast. To see that, build a synthetic daily series of London sales that ends yesterday: `80 + 7 x temperature + a weekend bump + noise`. Because the series is built from the temperature, the true effect is known.

In [ ]:
LONDON = "London, United Kingdom"


def make_sales(temperature: pd.Series, base: float, per_degree: float, seed: int = 42) -> pd.DataFrame:
    """Daily sales: a base level, a temperature effect, a weekend bump and noise."""
    rng = np.random.default_rng(seed)
    weekend = (temperature.index.dayofweek >= 5) * 15.0
    y = base + per_degree * temperature.to_numpy() + weekend + rng.normal(0, 10, len(temperature))
    return pd.DataFrame({"ds": temperature.index, "y": y})


yesterday = pd.Timestamp.now("UTC").tz_localize(None).normalize() - pd.Timedelta(days=1)
history_start = yesterday - pd.Timedelta(days=729)
measured = fetch_temperature(LONDON, history_start, yesterday).frame.set_index("ds")["temperature"]

sales = make_sales(measured, base=80, per_degree=7)
sales.tail()

Forecast 30 days with `weather=`. Open-Meteo's forecast API reaches 16 days ahead counting today, and the last of those is usually a partial day with no daily mean, so expect about 15 usable days (the table below shows the count). NeoProphet uses the weather forecast for them. **Climatology** fills the rest: the average temperature for that calendar day over the previous five full years. It is the typical weather for the date, not a prediction of it, so it carries the season but little of the day-to-day swing.

In [ ]:
result = forecast(sales, "30D", weather=LONDON)

by_source = result.weather.frame.groupby("source")["ds"].agg(["min", "max", "count"])
by_source

In [ ]:
fig = result.plot()

The credit line under the chart states how many days came from the weather forecast and how many from climatology. The effects chart ribbon shows the same split.

In [ ]:
fig = result.plot_regressor_effects()

To see the split in detail, here are the last 30 days of history and the horizon: the temperature the model was given, coloured by source, above the sales and the forecast.

In [ ]:
frame = result.predictions.assign(source=result.weather_sources)
frame["temperature_c"] = result.weather.frame.set_index("ds")["temperature"].reindex(frame["ds"]).to_numpy()
frame = frame.rename(columns={"temperature": "temperature_effect"})

last_day = sales["ds"].max()
recent = frame[frame["ds"] > last_day - pd.Timedelta(days=30)]
ahead = frame[frame["ds"] > last_day]
labels = {"archive": "measured", "forecast": "weather forecast", "climatology": "climatology"}
colours = {"archive": "tab:grey", "forecast": "tab:red", "climatology": "tab:blue"}

fig, (ax_t, ax_y) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, height_ratios=[1, 1.4])
ax_t.plot(recent["ds"], recent["temperature_c"], color="lightgrey", zorder=0)
for source, part in recent.groupby("source"):
    ax_t.scatter(part["ds"], part["temperature_c"], s=22, color=colours[source], label=labels[source])
ax_t.set_ylabel("Temperature (°C)")
ax_t.legend(loc="upper right", ncols=3, fontsize=8)

past = sales[sales["ds"] > last_day - pd.Timedelta(days=30)]
ax_y.plot(past["ds"], past["y"], "k.", label="actual")
ax_y.plot(recent["ds"], recent["yhat"], label="fitted and forecast")
ax_y.fill_between(ahead["ds"], ahead["yhat_lower"], ahead["yhat_upper"], alpha=0.2, label="80% interval")
for ax in (ax_t, ax_y):
    ax.axvline(last_day, color="k", linestyle=":", linewidth=1)
ax_y.set_ylabel("Sales")
ax_y.legend(loc="upper right", ncols=3, fontsize=8)
ax_y.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
fig.tight_layout()

**Takeaway:** the dotted line is the last observed day. Inside the weather forecast the sales forecast follows the temperature's day-to-day swings; under climatology the swings are smaller and the forecast mostly follows the seasonal drift. The table puts numbers on it.

In [ ]:
swing = ahead["temperature_effect"].diff().abs()
ahead.assign(day_to_day_change=swing).groupby("source").agg(
    days=("ds", "size"),
    mean_effect=("temperature_effect", "mean"),
    day_to_day_change=("day_to_day_change", "mean"),
).round(2)

How much does the weather buy here? Backtest the last 30 days with and without it. This is an upper bound: the held-out temperatures are measured values, whereas a live forecast has to use predicted ones.

In [ ]:
plain = forecast(sales, "30D")
pd.DataFrame({
    "without temperature": asdict(plain.backtest("30D").metrics),
    "with temperature": asdict(result.backtest("30D").metrics),
}).loc[["mae", "rmse", "coverage"]].T.round(3)

## 9. When a temperature regressor does not help

Temperature is only worth adding when the target really depends on it: ice cream, electricity demand, bike hire. For a series that ignores the weather, the regressor is just a smooth yearly curve with noise attached. To check, build sales with no temperature effect at all and add `weather=` anyway.

In [ ]:
flat_sales = make_sales(measured, base=170, per_degree=0.0, seed=7)
flat_plain = forecast(flat_sales, "30D")
flat_weather = forecast(flat_sales, "30D", weather=LONDON)

print(f"Fitted effect: {regressor_coefficients(flat_weather.model)['coef'].iloc[0]:+.2f} per °C (the true effect is 0)")
pd.DataFrame({
    "without temperature": asdict(flat_plain.backtest("30D").metrics),
    "with temperature": asdict(flat_weather.backtest("30D").metrics),
}).loc[["mae", "rmse", "coverage"]].T.round(3)

**Takeaway:** the fitted effect is close to zero and the errors are no better. A regressor that does not explain the target adds a chance of fitting noise, and for a live forecast it adds a network call and a data-source dependency. Add temperature when you have a reason, then check a backtest, as in sections 4, 7 and 8.

## Next

[05. Uncertainty and outliers](05_uncertainty_and_outliers.ipynb) looks at where the shaded bands come from, how much to trust them and what to do about bad data points.